# TF-IDF Baseline Models for Sentiment Classification

Comparison of Logistic Regression and MLP using different TF-IDF configurations.

## Import necessary libraries

In [7]:
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

## Load data and create train/validation split

In [8]:
DATA_DIR = Path("../data")

df = pd.read_json(DATA_DIR / "train.json")
test_df = pd.read_json(DATA_DIR / "test.json")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["sentiments"]
)
print(train_df.shape)

y_train = train_df["sentiments"]
y_val = val_df["sentiments"]

train_df.head()

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

(5920, 2)
Training samples: 5920
Validation samples: 1481
Test samples: 1851


## Define TF-IDF configurations and hyperparameters

In [9]:
tfidf_configs = [
    {
        "name": "Unigrams baseline",
        "ngram_range": (1, 1),
        "min_df": 1,
        "max_df": 1.0
    },
    {
        "name": "Unigrams + bigrams",
        "ngram_range": (1, 2),
        "min_df": 1,
        "max_df": 1.0,
    },
    {
        "name": "Unigrams + bigrams + filtering",
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
    }
]

C_values = [0.1, 1.0, 10.0]

mlp_configs = [
    {"hidden_layer_sizes": (64,), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (128, 64), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (256, 128), "learning_rate_init": 0.001}
]

## Logistic Regression

In [10]:
lr_results = []
best_lr_f1 = -1
best_lr_model = None
best_vectorizer = None
best_lr_config = None

for config in tfidf_configs:
    print(f"\nTesting TF-IDF: {config['name']}")

    vectorizer = TfidfVectorizer(
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        max_df=config["max_df"]
    )

    # Fit TF-IDF ONLY on training reviews
    X_train = vectorizer.fit_transform(train_df["reviews"])
    X_val = vectorizer.transform(val_df["reviews"])

    y_train = train_df["sentiments"]
    y_val = val_df["sentiments"]

    for C in C_values:
        model = LogisticRegression(C=C, max_iter=1000, random_state=42)

        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        accuracy = accuracy_score(y_val, predictions)
        precision = precision_score(y_val, predictions, zero_division=0)
        recall = recall_score(y_val, predictions, zero_division=0)
        f1 = f1_score(y_val, predictions, zero_division=0)

        result = {
            "TF-IDF": config["name"],
            "C": C,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        }
        lr_results.append(result)

        print(
            f"C={C:<4} | Accuracy={accuracy:.4f} "
            f"| Precision={precision:.4f} "
            f"| Recall={recall:.4f} | F1={f1:.4f}"
        )

        if f1 > best_lr_f1:
            best_lr_f1 = f1
            best_lr_model = model
            best_vectorizer = vectorizer
            best_lr_config = result.copy()

lr_results_df = pd.DataFrame(lr_results).sort_values("F1", ascending=False)
print("Logistic Regression: All Results")
display(lr_results_df)

print("\nBEST LOGISTIC REGRESSION CONFIGURATION")
print(best_lr_config)

X_val_best = best_vectorizer.transform(val_df["reviews"])
best_lr_predictions = best_lr_model.predict(X_val_best)

print("\nClassification report:")
print(classification_report(y_val, best_lr_predictions))

print("Confusion matrix:")
print(confusion_matrix(y_val, best_lr_predictions))


Testing TF-IDF: Unigrams baseline
C=0.1  | Accuracy=0.8535 | Precision=0.8535 | Recall=1.0000 | F1=0.9209
C=1.0  | Accuracy=0.9007 | Precision=0.8998 | Recall=0.9945 | F1=0.9448
C=10.0 | Accuracy=0.9217 | Precision=0.9342 | Recall=0.9771 | F1=0.9551

Testing TF-IDF: Unigrams + bigrams
C=0.1  | Accuracy=0.8535 | Precision=0.8535 | Recall=1.0000 | F1=0.9209
C=1.0  | Accuracy=0.8812 | Precision=0.8778 | Recall=1.0000 | F1=0.9349
C=10.0 | Accuracy=0.9190 | Precision=0.9194 | Recall=0.9921 | F1=0.9543

Testing TF-IDF: Unigrams + bigrams + filtering
C=0.1  | Accuracy=0.8535 | Precision=0.8535 | Recall=1.0000 | F1=0.9209
C=1.0  | Accuracy=0.8893 | Precision=0.8857 | Recall=0.9992 | F1=0.9390
C=10.0 | Accuracy=0.9217 | Precision=0.9252 | Recall=0.9881 | F1=0.9556
Logistic Regression: All Results


,TF-IDF,C,Accuracy,Precision,Recall,F1
8,Unigrams + bigrams + filtering,10.0,0.921675,0.925185,0.988133,0.955624
2,Unigrams baseline,10.0,0.921675,0.934191,0.977057,0.955143
5,Unigrams + bigrams,10.0,0.918974,0.919355,0.992089,0.954338
1,Unigrams baseline,1.0,0.900743,0.899785,0.994462,0.944758
7,Unigrams + bigrams + filtering,1.0,0.889264,0.885694,0.999209,0.939033
4,Unigrams + bigrams,1.0,0.881161,0.877778,1.000000,0.934911
0,Unigrams baseline,0.1,0.853477,0.853477,1.000000,0.920947
3,Unigrams + bigrams,0.1,0.853477,0.853477,1.000000,0.920947
6,Unigrams + bigrams + filtering,0.1,0.853477,0.853477,1.000000,0.920947



BEST LOGISTIC REGRESSION CONFIGURATION
{'TF-IDF': 'Unigrams + bigrams + filtering', 'C': 10.0, 'Accuracy': 0.9216745442268738, 'Precision': 0.9251851851851852, 'Recall': 0.9881329113924051, 'F1': 0.9556235654169855}

Classification report:
              precision    recall  f1-score   support

           0       0.89      0.53      0.67       217
           1       0.93      0.99      0.96      1264

    accuracy                           0.92      1481
   macro avg       0.91      0.76      0.81      1481
weighted avg       0.92      0.92      0.91      1481

Confusion matrix:
[[ 116  101]
 [  15 1249]]


## Multi-Layer Perceptron (MLP)

In [11]:
mlp_results = []
best_mlp_f1 = -1
best_mlp_model = None
best_mlp_vectorizer = None
best_mlp_config = None

for tfidf_config in tfidf_configs:
    vectorizer = TfidfVectorizer(
        ngram_range=tfidf_config["ngram_range"],
        min_df=tfidf_config["min_df"],
        max_df=tfidf_config["max_df"]
    )

    X_train_mlp = vectorizer.fit_transform(train_df["reviews"])
    X_val_mlp = vectorizer.transform(val_df["reviews"])

    for config in mlp_configs:
        print(
            f"\nTesting MLP: TF-IDF={tfidf_config['name']}, "
            f"layers={config['hidden_layer_sizes']}, "
            f"learning_rate={config['learning_rate_init']}"
        )

        model = MLPClassifier(
            hidden_layer_sizes=config["hidden_layer_sizes"],
            activation="relu",
            solver="adam",
            learning_rate_init=config["learning_rate_init"],
            batch_size=64,
            max_iter=50,
            early_stopping=True,
            n_iter_no_change=5,
            random_state=42
        )

        model.fit(X_train_mlp, y_train)
        predictions = model.predict(X_val_mlp)

        accuracy = accuracy_score(y_val, predictions)
        precision = precision_score(y_val, predictions, zero_division=0)
        recall = recall_score(y_val, predictions, zero_division=0)
        f1 = f1_score(y_val, predictions, zero_division=0)

        result = {
            "TF-IDF": tfidf_config["name"],
            "Hidden layers": str(config["hidden_layer_sizes"]),
            "Learning rate": config["learning_rate_init"],
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        }
        mlp_results.append(result)

        print(
            f"Accuracy={accuracy:.4f} | Precision={precision:.4f} "
            f"| Recall={recall:.4f} | F1={f1:.4f}"
        )

        if f1 > best_mlp_f1:
            best_mlp_f1 = f1
            best_mlp_model = model
            best_mlp_vectorizer = vectorizer
            best_mlp_config = result.copy()

mlp_results_df = pd.DataFrame(mlp_results).sort_values(
    "F1", ascending=False
)

print("\n MLP: ALL RESULTS ")
print(mlp_results_df.to_string(index=False))

print("\n BEST MLP CONFIGURATION ")
print(best_mlp_config)

X_val_mlp = best_mlp_vectorizer.transform(val_df["reviews"])
best_mlp_predictions = best_mlp_model.predict(X_val_mlp)

print("\n Classification report:")
print(classification_report(y_val, best_mlp_predictions))

print("Confusion matrix:")
print(confusion_matrix(y_val, best_mlp_predictions))


Testing MLP: TF-IDF=Unigrams baseline, layers=(64,), learning_rate=0.001
Accuracy=0.9183 | Precision=0.9339 | Recall=0.9731 | F1=0.9531

Testing MLP: TF-IDF=Unigrams baseline, layers=(128, 64), learning_rate=0.001
Accuracy=0.9190 | Precision=0.9511 | Recall=0.9541 | F1=0.9526

Testing MLP: TF-IDF=Unigrams baseline, layers=(256, 128), learning_rate=0.001
Accuracy=0.9163 | Precision=0.9378 | Recall=0.9660 | F1=0.9517

Testing MLP: TF-IDF=Unigrams + bigrams, layers=(64,), learning_rate=0.001
Accuracy=0.9223 | Precision=0.9240 | Recall=0.9905 | F1=0.9561

Testing MLP: TF-IDF=Unigrams + bigrams, layers=(128, 64), learning_rate=0.001
Accuracy=0.9264 | Precision=0.9365 | Recall=0.9802 | F1=0.9579

Testing MLP: TF-IDF=Unigrams + bigrams, layers=(256, 128), learning_rate=0.001
Accuracy=0.9305 | Precision=0.9435 | Recall=0.9771 | F1=0.9600

Testing MLP: TF-IDF=Unigrams + bigrams + filtering, layers=(64,), learning_rate=0.001
Accuracy=0.9278 | Precision=0.9353 | Recall=0.9834 | F1=0.9587

Testin

In [13]:
import joblib

baseline_data = {
    "lr_model": best_lr_model,
    "lr_vectorizer": best_vectorizer,
    "mlp": best_mlp_model,
    "mlp_vectorizer": best_mlp_vectorizer,
    "val_df": val_df.reset_index(drop=True)
}

joblib.dump(baseline_data, "baseline_models.pkl")

print("Baseline models and validation data saved.")

Baseline models and validation data saved.
